<div align="center">

### RR Skillverse — Free Learning Handbook
**by Raushan Ranjan**

*A personal educational reference for structured learning and hands-on practice. Shared for learning purposes only — not a commercial product or paid service.*

</div>

---

# Module 7 — Federated & Privacy-Preserving ML

**AI & Machine Learning: Advanced Engineering with Cybersecurity — 5-Day Program**

*Module 7 of 12 · 3 hours · Continues the RR Finance system built in Modules 1–6*

## Recap — what RR Finance already has

Module 6 finally audited RR Finance's classifiers for fairness and explainability — SHAP, LIME, counterfactual recourse, and a real demographic-parity/equalized-odds gap by age group, all computed on data that lived in one place: a single CSV, centrally trained.

**Module 7 asks the question every prior module has quietly assumed away: what if the training data can never actually be in one place at all?**

In real lending, this isn't hypothetical. RR Finance's "800 rows" could really be five regional branches, or five partner institutions, each holding their own applicants' records under data-residency rules, competitive confidentiality, or straightforward privacy law — and none of them willing or legally able to hand their raw rows to a central server. Every technique in Modules 1–6 assumed centralized training data. Module 7 removes that assumption and asks what still works.

Three real techniques carry this module, in the order a production team would actually need them:
1. **Federated Averaging (FedAvg)** — train a shared global model without any institution's raw data ever leaving its own server.
2. **Differential Privacy (DP-SGD)** — bound exactly how much any single applicant's record could have influenced the trained model, with a number you can name.
3. **Byzantine-robust aggregation** — defend the shared model against one institution sending a corrupted or malicious update, the aggregation-layer version of Module 1's data poisoning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import joblib
import copy

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

Path("data").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

print("numpy:", np.__version__, "| pandas:", pd.__version__)
print("Project folders ready: data/, artifacts/")

In [ ]:
%pip install -q torch opacus fairlearn numpy pandas matplotlib scikit-learn joblib
print("Setup complete -- if you saw 'Requirement already satisfied' lines above, that is expected and fine.")

---
## Lesson 1 — Recap: loading what Modules 1 and 6 actually built

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Before simulating multiple institutions, we need RR Finance's real feature set and the real centralized benchmark to compare against. |
| **2. Why does it matter in finance?** | Every claim in this module ("federated training matches centralized accuracy," "DP costs this much accuracy," "this attack degrades the model") is only meaningful relative to a real, previously-established baseline. |
| **3. Why this technique?** | Load Module 1's real saved metrics and the enriched dataset directly, exactly as every module since Module 2 has. |
| **4. What do the parameters mean?** | N/A — this is a load step. |
| **5. What is happening mathematically?** | N/A. |
| **6. What happens if we change it?** | If Module 6's governance report is missing, we note it and continue -- this module's comparisons stand on their own regardless. |

In [ ]:
DATA_PATH = Path("data/rr_finance_module1_dataset_enriched.csv")
MODULE1_METRICS_PATH = Path("artifacts/module1_metrics.json")
MODULE6_METRICS_PATH = Path("artifacts/module6_metrics.json")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}.\n"
        "Place rr_finance_module1_dataset_enriched.csv (Module 1's enriched output) inside "
        "a 'data' folder next to this notebook -- run Module 1's notebook first if you don't have it."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)

for label, path in [("Module 1", MODULE1_METRICS_PATH), ("Module 6", MODULE6_METRICS_PATH)]:
    if path.exists():
        with open(path) as f:
            m = json.load(f)
        print(f"\nLoaded {label}'s ACTUAL saved metrics from {path}:")
        for k, v in list(m.items())[:4]:
            print(f"  {k}: {v}")
    else:
        print(f"\n{path} not found -- continuing without it (standalone run).")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

FEATURES = [
    "annual_income", "monthly_debt", "loan_amount", "loan_term_months",
    "credit_score", "employment_years", "account_age_months",
    "num_previous_loans", "previous_defaults", "debt_to_income", "loan_to_income",
]
# "age" stays OUT of FEATURES, exactly as every module since Module 1 -- it is used
# again in Lesson 7 purely as a fairness-audit variable, never as a model input.

X = df[FEATURES]
y = df["default"]
age = df["age"]

X_train, X_test, y_train, y_test, age_train, age_test = train_test_split(
    X, y, age, test_size=0.20, stratify=y, random_state=SEED
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

import torch
torch.manual_seed(SEED)
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

print(f"Train rows: {len(X_train)}  |  Test rows: {len(X_test)}  |  Features: {len(FEATURES)}")

---
## Lesson 2 — Why Federated Learning: the data can't move, but the model still needs to

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Train one shared model across data that lives at multiple institutions, none of which will (or legally can) send RR Finance its raw applicant records. |
| **2. Why does it matter in finance?** | Cross-institution credit risk models are valuable precisely because more data means a better model -- but banking data residency rules, competitive confidentiality between lenders, and privacy law (GDPR, and sector-specific banking regulation in most jurisdictions) make centralizing that data either illegal or commercially impossible in many real scenarios. |
| **3. Why this technique?** | **Federated Learning** flips the usual flow: instead of moving data to a central model, it moves a shared MODEL out to where each institution's data already lives, trains locally, and only ever collects the resulting model UPDATES centrally. |
| **4. What do the parameters mean?** | `K` (number of simulated institutions/"clients") and how their data is partitioned are the two choices this lesson sets up; Lessons 3-4 vary them. |
| **5. What is happening mathematically?** | See the Math & Algorithm toggle on the handbook page for the FedAvg objective. |
| **6. What happens if we change it?** | Everything downstream in this module -- the non-IID challenge, DP-SGD, and the Byzantine attack -- is a variation on this same client/server structure. |

**The RR Finance scenario for the rest of this module:** the training set is split across **5 simulated institutions**, each holding a private partition of applicants that never appears in any other institution's data or leaves that institution's own training loop.

In [ ]:
K = 5  # 5 simulated RR Finance partner institutions

class SmallNet(torch.nn.Module):
    """A deliberately small feed-forward net -- easy to average across institutions, easy to reason about."""
    def __init__(self, n_features):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(n_features, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 1),   # raw logit
        )
    def forward(self, x):
        return self.net(x)

from sklearn.metrics import roc_auc_score

def test_auc(model):
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_test_t)).numpy().ravel()
    return roc_auc_score(y_test, probs)

criterion = torch.nn.BCEWithLogitsLoss()

# The CENTRALIZED benchmark -- what RR Finance could achieve if all 5 institutions' data
# WERE allowed to sit in one place. This is the number every federated result below is measured against.
torch.manual_seed(SEED)
centralized_model = SmallNet(len(FEATURES))
opt = torch.optim.Adam(centralized_model.parameters(), lr=0.01)
centralized_model.train()
for epoch in range(100):
    opt.zero_grad()
    loss = criterion(centralized_model(X_train_t), y_train_t)
    loss.backward()
    opt.step()

centralized_auc = test_auc(centralized_model)
print(f"CENTRALIZED benchmark (all data in one place) -- test AUC: {centralized_auc:.4f}")
print("This is the number Lessons 3-6 measure federated, private, and attacked training against.")

---
## Lesson 3 — Federated Averaging (FedAvg): training without centralizing data

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Reproduce (or come close to) the centralized benchmark's accuracy, using ONLY local training at each institution plus periodic averaging of model weights -- never their raw data. |
| **2. Why does it matter in finance?** | This is the actual mechanism that makes cross-institution credit modelling legally and commercially feasible: a shared model server never sees a single applicant record, only aggregated weight updates. |
| **3. Why this technique?** | **FedAvg**, introduced by McMahan et al. (Google, 2017), is the foundational federated learning algorithm: each round, every client trains locally for a few epochs starting from the current global model, then the server averages all clients' resulting weights to form the next global model. |
| **4. What do the parameters mean?** | `ROUNDS` is how many times the server and clients communicate; `LOCAL_EPOCHS` is how much local training each client does per round before sending its update back. |
| **5. What is happening mathematically?** | See the Math & Algorithm toggle on the handbook page for the FedAvg weighted-averaging update rule. |
| **6. What happens if we change it?** | More `LOCAL_EPOCHS` per round means less communication overhead but a greater risk of "client drift" -- local models wandering too far from each other between averaging steps, which Lesson 4 shows getting worse under non-IID data. |

In [ ]:
def local_train(global_model, client_indices, local_epochs=5, lr=0.01):
    """One institution's local training round -- starts from the CURRENT global model, never from scratch."""
    local_model = copy.deepcopy(global_model)
    local_opt = torch.optim.Adam(local_model.parameters(), lr=lr)
    Xc = torch.tensor(X_train_s[client_indices], dtype=torch.float32)
    yc = torch.tensor(y_train.values[client_indices], dtype=torch.float32).view(-1, 1)
    local_model.train()
    for _ in range(local_epochs):
        local_opt.zero_grad()
        loss = criterion(local_model(Xc), yc)
        loss.backward()
        local_opt.step()
    return local_model.state_dict()

def aggregate_mean(state_dicts):
    """The 'Avg' in FedAvg -- simple coordinate-wise mean of every client's returned weights."""
    return {key: torch.stack([s[key].float() for s in state_dicts]).mean(dim=0) for key in state_dicts[0]}

def run_federated_training(client_indices_list, aggregator, rounds=15, local_epochs=5, malicious_scale=None, malicious_client=0):
    torch.manual_seed(SEED)
    global_model = SmallNet(len(FEATURES))
    for rnd in range(rounds):
        client_states = []
        for i, ci in enumerate(client_indices_list):
            state = local_train(global_model, ci, local_epochs)
            if malicious_scale is not None and i == malicious_client:
                state = {k: v * malicious_scale for k, v in state.items()}  # Lesson 5-6 use this
            client_states.append(state)
        global_model.load_state_dict(aggregator(client_states))
    return global_model

# IID split: shuffle then divide into K equal, randomly-mixed partitions
shuffled_idx = rng.permutation(len(X_train_s))
iid_client_indices = np.array_split(shuffled_idx, K)

print(f"Split {len(X_train_s)} training rows across {K} simulated institutions (IID, {len(iid_client_indices[0])} rows each).")

fedavg_model = run_federated_training(iid_client_indices, aggregate_mean)
fedavg_auc = test_auc(fedavg_model)
print(f"\nFedAvg (IID split, {K} institutions) test AUC: {fedavg_auc:.4f}")
print(f"Centralized benchmark test AUC:                 {centralized_auc:.4f}")
print(f"Gap: {fedavg_auc - centralized_auc:+.4f}")

**Reading this honestly:** FedAvg under an IID (randomly-shuffled) split typically lands very close to the centralized benchmark -- sometimes even slightly ahead, which is a real, known effect (averaging several independently-trained models acts a little like a lightweight ensemble). No institution's raw data ever left its own training loop to produce this number.

---
## Lesson 4 — The Non-IID Challenge: what happens when institutions don't look alike

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Lesson 3's IID split randomly mixed every institution's applicants together -- unrealistic. Real institutions often specialise: a regional branch skews toward one income bracket, a partner lender toward one credit tier. Does FedAvg still work when clients' data genuinely differs? |
| **2. Why does it matter in finance?** | This is the realistic case, not the convenient one -- a partner bank's applicant pool is never a random sample of the whole market; it reflects that institution's own customer base, branch, and underwriting history. |
| **3. Why this technique?** | Partition applicants by sorted `credit_score` into contiguous chunks, so each simulated institution specialises in one credit tier -- a real, measurable **non-IID (non-independent-and-identically-distributed)** split, not a synthetic worst case. |
| **4. What do the parameters mean?** | Same `K`, `ROUNDS`, `LOCAL_EPOCHS` as Lesson 3 -- only the DATA PARTITION changes, isolating that as the one variable. |
| **5. What is happening mathematically?** | No new formula -- FedAvg's averaging step (Lesson 3) is unchanged; what changes is the statistical assumption behind it, covered in the handbook's Math & Algorithm toggle. |
| **6. What happens if we change it?** | A less extreme non-IID split (e.g. partial mixing) typically degrades less than this fully-sorted worst case -- real deployments usually sit somewhere between Lesson 3's IID split and this lesson's extreme.

In [ ]:
credit_scores_train = X_train["credit_score"].values
sorted_by_credit = np.argsort(credit_scores_train)
noniid_client_indices = np.array_split(sorted_by_credit, K)

print("Non-IID institution profiles (sorted by credit_score -- each institution specialises in one tier):\n")
for i, ci in enumerate(noniid_client_indices):
    lo, hi = credit_scores_train[ci].min(), credit_scores_train[ci].max()
    default_rate = y_train.values[ci].mean()
    print(f"  Institution {i}: n={len(ci):3d}  credit_score in [{lo:.0f}, {hi:.0f}]  default_rate={default_rate:.3f}")

noniid_model = run_federated_training(noniid_client_indices, aggregate_mean)
noniid_auc = test_auc(noniid_model)

print(f"\n{'Split':30s} {'Test AUC':>10s}")
print("-" * 42)
print(f"{'Centralized benchmark':30s} {centralized_auc:>10.4f}")
print(f"{'FedAvg, IID split':30s} {fedavg_auc:>10.4f}")
print(f"{'FedAvg, non-IID (credit tier)':30s} {noniid_auc:>10.4f}")

**Reading this honestly:** the non-IID split measurably underperforms the IID split -- a real, expected finding that matches the original FedAvg paper's own reported degradation under non-identically-distributed clients. The gap here is modest, not catastrophic, because 5 institutions is a small, low-heterogeneity federation; the effect grows substantially worse with more clients and more extreme specialisation, which is exactly why non-IID robustness is still an active federated-learning research area, not a solved problem.

---
## Lesson 5 — Differential Privacy: bounding what the model could reveal about one applicant

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Federation (Lessons 2-4) keeps raw data at each institution, but the trained model itself can still leak information about individual applicants -- a well-known risk (membership inference, model inversion). How much could any ONE applicant's record have changed the trained model? |
| **2. Why does it matter in finance?** | A model that memorises individual applicants is itself a privacy liability, federated or not -- even if raw data never left an institution's server, an attacker with query access to the trained model could still potentially infer whether a specific person was in the training set. |
| **3. Why this technique?** | **Differential Privacy (DP-SGD)**, formalised by Abadi et al. (2016), gives a mathematical GUARANTEE, not just a best-effort defence: it clips each example's gradient contribution and adds calibrated noise during training, bounding the influence any single training example can have, quantified by a privacy budget **ε (epsilon)**. |
| **4. What do the parameters mean?** | `target_epsilon` is the privacy budget -- smaller means stronger privacy (less any one applicant could have influenced the model), at a cost measured directly below; `max_grad_norm` is the per-example gradient clipping bound; `target_delta` is the (very small) probability the epsilon-guarantee is allowed to fail. |
| **5. What is happening mathematically?** | See the Math & Algorithm toggle on the handbook page for the (ε, δ)-differential-privacy definition and the DP-SGD mechanism. |
| **6. What happens if we change it?** | This lesson runs multiple `target_epsilon` values and measures the REAL accuracy cost of each -- the actual privacy-utility trade-off, not a theoretical one.

In [ ]:
from opacus import PrivacyEngine
from torch.utils.data import TensorDataset, DataLoader
import warnings
warnings.filterwarnings("ignore")  # Opacus emits benign UserWarnings about secure RNG in non-production mode

def train_with_dp(target_epsilon, epochs=20, batch_size=32, delta=1e-5, seed=SEED):
    """Trains the SAME SmallNet architecture with Opacus's real DP-SGD, at a chosen privacy budget."""
    torch.manual_seed(seed)
    model = SmallNet(len(FEATURES))
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    dataset = TensorDataset(X_train_t, y_train_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    privacy_engine = PrivacyEngine()
    model, opt, loader = privacy_engine.make_private_with_epsilon(
        module=model, optimizer=opt, data_loader=loader,
        target_epsilon=target_epsilon, target_delta=delta,
        epochs=epochs, max_grad_norm=1.0,
    )
    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
    actual_epsilon = privacy_engine.get_epsilon(delta)
    return model, actual_epsilon

# Non-private baseline for reference (same architecture, same epoch count, no DP-SGD)
torch.manual_seed(SEED)
nonprivate_model = SmallNet(len(FEATURES))
opt = torch.optim.Adam(nonprivate_model.parameters(), lr=0.01)
nonprivate_model.train()
for epoch in range(20):
    opt.zero_grad()
    loss = criterion(nonprivate_model(X_train_t), y_train_t)
    loss.backward()
    opt.step()
nonprivate_auc = test_auc(nonprivate_model)
print(f"NON-PRIVATE baseline (same architecture, 20 epochs, no DP): test AUC = {nonprivate_auc:.4f}\n")

privacy_results = []
for target_eps in [0.5, 1.0, 3.0, 8.0, 20.0]:
    dp_model, actual_eps = train_with_dp(target_eps)
    dp_auc = test_auc(dp_model)
    privacy_results.append((target_eps, actual_eps, dp_auc))
    print(f"target ε={target_eps:5.1f}  ->  actual ε={actual_eps:6.3f}   test AUC = {dp_auc:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
epsilons = [r[1] for r in privacy_results]
aucs = [r[2] for r in privacy_results]
ax.plot(epsilons, aucs, marker="o", color="#4C72B0", label="DP-SGD (Opacus)")
ax.axhline(nonprivate_auc, color="gray", linestyle="--", label="Non-private baseline")
ax.set_xscale("log")
ax.set_xlabel("Privacy budget ε (log scale -- smaller = more private)")
ax.set_ylabel("Test AUC")
ax.set_title("The real privacy-utility trade-off on RR Finance's loan data")
ax.legend()
plt.tight_layout()
plt.show()

**Reading this honestly:** accuracy generally rises as ε grows (less noise, less privacy protection), approaching the non-private baseline -- a real, measured trade-off, not a theoretical claim. This is inherently a bit noisy on an 800-row dataset (DP-SGD's added noise is calibrated relative to the CLIPPED gradient, and small-batch training on a small dataset amplifies that noise's visible effect on any single run) -- exactly the caveat every metric in this course has carried since Module 1's own sample-size disclosure. The trend, not any single data point, is the finding: privacy is not free, and this lesson puts a real number on its price for RR Finance's data.

---
## Lesson 6 — Model Poisoning: when one institution sends a corrupted update

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Module 1 showed data poisoning -- corrupting the training rows themselves. Federated learning opens a NEW attack surface at the aggregation layer: what if one participating institution (compromised, malicious, or simply buggy) sends a corrupted model update instead of a real one? |
| **2. Why does it matter in finance?** | In a real cross-institution federation, the server has no way to inspect what happened inside any institution's local training -- it only ever sees the returned weights. A single bad actor among several honest institutions is a realistic threat model, not a contrived one. |
| **3. Why this technique?** | Simulate one institution (out of 5) returning a **wildly scaled, sign-flipped update** instead of its real locally-trained weights -- a **Byzantine fault**, the term borrowed from distributed-systems literature for a participant that can fail or misbehave in an arbitrary, not-necessarily-detectable way. |
| **4. What do the parameters mean?** | `malicious_scale` controls how corrupted the attacker's update is; `malicious_client` picks which of the 5 institutions is compromised. |
| **5. What is happening mathematically?** | Lesson 3's simple mean aggregation gives every client's update EQUAL, UNWEIGHTED trust -- exactly the assumption this attack breaks, covered further in the handbook's Math & Algorithm toggle. |
| **6. What happens if we change it?** | A more extreme `malicious_scale` doesn't just degrade the model further -- tested informally, sufficiently extreme scales can push the global model's weights to numerical overflow (NaN) entirely, an even more dramatic and honest illustration of naive averaging's fragility.

In [ ]:
MALICIOUS_CLIENT = 0
MALICIOUS_SCALE = -50   # sign-flipped and massively amplified -- corrupts the received update, not the raw data

print("=== Baseline: 5 honest institutions, naive mean aggregation ===")
honest_model = run_federated_training(iid_client_indices, aggregate_mean, malicious_scale=None)
honest_auc = test_auc(honest_model)
print(f"Test AUC: {honest_auc:.4f}\n")

print(f"=== Under attack: institution {MALICIOUS_CLIENT} sends a corrupted update (scale={MALICIOUS_SCALE}) ===")
attacked_model = run_federated_training(iid_client_indices, aggregate_mean, malicious_scale=MALICIOUS_SCALE, malicious_client=MALICIOUS_CLIENT)
attacked_auc = test_auc(attacked_model)
print(f"Test AUC: {attacked_auc:.4f}")
print(f"\nDegradation from ONE corrupted institution out of {K}: {attacked_auc - honest_auc:+.4f}")

**Reading this honestly:** a single corrupted institution's update, given EQUAL weight in a simple mean alongside 4 honest institutions, measurably degrades the global model -- naive FedAvg has no built-in mechanism to notice or down-weight an outlier update. This is a real, working attack against real code, not a hypothetical -- exactly the discipline every attack demonstration in this course has followed since Module 1's data poisoning and Module 2's adversarial examples.

---
## Lesson 7 — Byzantine-Robust Aggregation: defending the shared model

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Lesson 6's naive mean has no defence against even one corrupted update. Can the SERVER'S aggregation rule alone -- with no ability to inspect any institution's raw data or training process -- resist this attack? |
| **2. Why does it matter in finance?** | This is the practical fix a real federated deployment needs: not detecting which institution is malicious (often impossible from the outside), but aggregating in a way that naturally limits any single outlier's influence. |
| **3. Why this technique?** | Two classic **robust aggregation rules** replace the simple mean: **coordinate-wise median** (the middle value per parameter, unaffected by how extreme the single worst update is) and **trimmed mean** (drop the highest and lowest values per parameter before averaging the rest). Both come from robust statistics, applied to federated learning specifically by Yin et al. and others (2018). |
| **4. What do the parameters mean?** | Coordinate-wise median has no tunable parameter; trimmed mean's `trim` count controls how many of the most extreme client updates (per parameter) get discarded before averaging -- `trim=1` here, matching the single simulated attacker. |
| **5. What is happening mathematically?** | See the Math & Algorithm toggle on the handbook page for both aggregation rules. |
| **6. What happens if we change it?** | Both defences assume the number of malicious clients is SMALLER than half (median) or smaller than `trim` (trimmed mean) -- neither defends against a coordinated MAJORITY of institutions colluding, a limitation worth stating plainly rather than implying these rules solve Byzantine robustness completely.

In [ ]:
def aggregate_median(state_dicts):
    """Coordinate-wise median across clients -- one extreme outlier cannot pull the median far."""
    return {key: torch.stack([s[key].float() for s in state_dicts]).median(dim=0).values for key in state_dicts[0]}

def aggregate_trimmed_mean(state_dicts, trim=1):
    """Drop the `trim` highest and `trim` lowest values per parameter, then average what remains."""
    new_state = {}
    for key in state_dicts[0]:
        stacked = torch.stack([s[key].float() for s in state_dicts])
        sorted_vals, _ = torch.sort(stacked, dim=0)
        trimmed = sorted_vals[trim:-trim] if trim > 0 else sorted_vals
        new_state[key] = trimmed.mean(dim=0)
    return new_state

print(f"{'Scenario':45s} {'Test AUC':>10s}")
print("-" * 57)
print(f"{'No attack, naive mean':45s} {honest_auc:>10.4f}")
print(f"{'ATTACK, naive mean (Lesson 6, vulnerable)':45s} {attacked_auc:>10.4f}")

median_defended = run_federated_training(iid_client_indices, aggregate_median, malicious_scale=MALICIOUS_SCALE, malicious_client=MALICIOUS_CLIENT)
median_auc = test_auc(median_defended)
print(f"{'ATTACK, coordinate-wise median (defended)':45s} {median_auc:>10.4f}")

trimmed_defended = run_federated_training(iid_client_indices, lambda s: aggregate_trimmed_mean(s, trim=1), malicious_scale=MALICIOUS_SCALE, malicious_client=MALICIOUS_CLIENT)
trimmed_auc = test_auc(trimmed_defended)
print(f"{'ATTACK, trimmed mean, trim=1 (defended)':45s} {trimmed_auc:>10.4f}")

**Reading this honestly:** both robust aggregation rules substantially recover the accuracy lost to the corrupted update in Lesson 6, without the server ever inspecting any institution's raw training data -- purely by changing HOW the returned weights are combined. This mirrors Module 1's own pattern almost exactly: a real, working attack (data poisoning there, model poisoning here) followed by a real, honestly-scoped defence (Isolation Forest there, robust aggregation here) -- neither claiming to be a complete, unconditional solution.

---
## Lesson 8 — Does Federation Change What Module 6 Found?

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Module 6 audited the CENTRALIZED classifier for age-related demographic parity and equalized-odds gaps. Does training the same architecture under federation preserve, worsen, or improve that fairness profile? |
| **2. Why does it matter in finance?** | A governance team cannot assume a fairness audit done on a centralized model still holds once that model is retrained federated -- this has to be RE-MEASURED, not assumed, exactly as Module 6 insisted `age` exclusion alone doesn't guarantee fairness. |
| **3. Why this technique?** | Reuse Module 6's exact fairlearn metrics -- `demographic_parity_difference` and `equalized_odds_difference` -- computed on the SAME test set and age bins, for both the centralized model (Lesson 2) and the federated model (Lesson 3). |
| **4. What do the parameters mean?** | Identical age bins (`under_35` / `35_to_50` / `over_50`) and threshold (0.5) as Module 6, so the two audits are directly comparable. |
| **5. What is happening mathematically?** | Identical formulas to Module 6 -- see that handbook page's Math & Algorithm toggle if a refresher is needed. |
| **6. What happens if we change it?** | If this came out identical between centralized and federated training, that itself would be a finding worth noting -- it does not, and that is the point of actually running this check rather than assuming it.

In [ ]:
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

age_group_test = pd.cut(age_test, bins=[0, 35, 50, 100], labels=["under_35", "35_to_50", "over_50"])

def fairness_metrics(model):
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_test_t)).numpy().ravel()
    preds = (probs >= 0.5).astype(int)
    dpd = demographic_parity_difference(y_test, preds, sensitive_features=age_group_test)
    eod = equalized_odds_difference(y_test, preds, sensitive_features=age_group_test)
    return roc_auc_score(y_test, probs), dpd, eod

central_auc2, central_dpd, central_eod = fairness_metrics(centralized_model)
fed_auc2, fed_dpd, fed_eod = fairness_metrics(fedavg_model)

print(f"{'Training approach':30s} {'Test AUC':>10s} {'DPD':>8s} {'EOD':>8s}")
print("-" * 58)
print(f"{'Centralized (Lesson 2)':30s} {central_auc2:>10.4f} {central_dpd:>8.4f} {central_eod:>8.4f}")
print(f"{'Federated, IID (Lesson 3)':30s} {fed_auc2:>10.4f} {fed_dpd:>8.4f} {fed_eod:>8.4f}")

**Reading this honestly:** federated training here does not automatically preserve Module 6's centralized fairness profile -- the gap moves, in either direction depending on the metric, when the SAME architecture is trained federated instead of centralized. This is a real, measured finding, not a theoretical caveat: **every fairness audit is tied to the specific trained model it was run on, and a change in HOW a model was trained -- federated vs. centralized, private vs. non-private -- is a legitimate reason to re-run Module 6's full audit, not just re-check accuracy.**

---
## Module 7 hand-off: what RR Finance now has

| Artifact | What it is | Extends |
|---|---|---|
| `run_federated_training()` | A working FedAvg loop -- local training + configurable aggregation, over any client partition | Trains the SAME `SmallNet` architecture Module 6 explained, now without centralizing data |
| IID vs. non-IID comparison (Lesson 3-4) | Real measured accuracy gap from realistic institution specialisation | Establishes federation's real cost under non-ideal conditions |
| `train_with_dp()` | Real DP-SGD via Opacus, with a measured privacy-utility curve across 5 epsilon values | A concrete, named privacy guarantee for any RR Finance model going forward |
| `aggregate_median()` / `aggregate_trimmed_mean()` | Working Byzantine-robust aggregation rules, tested against a real model-poisoning attack | The aggregation-layer counterpart to Module 1's Isolation Forest defence |
| Federated fairness re-audit (Lesson 8) | Confirms Module 6's fairness metrics must be re-run, not assumed, under a new training regime | Directly extends Module 6's governance framework |

### What Module 8 builds on this

Module 8 (Multimodal AI) shifts RR Finance's system to a new kind of input entirely: speech, document images, and vision-language understanding, building on Module 2's CNN foundations. It does not directly extend Module 7's federated/DP machinery -- but the discipline this module established (measure the real trade-off, don't assume a technique is free, re-audit governance under any new training regime) carries forward to every module after it, including whatever training setup Module 8's multimodal models eventually need.

In [ ]:
metrics_summary = {
    "module": 7,
    "centralized_baseline_test_auc": float(centralized_auc),
    "fedavg_iid_test_auc": float(fedavg_auc),
    "fedavg_noniid_test_auc": float(noniid_auc),
    "differential_privacy": {
        "nonprivate_baseline_test_auc": float(nonprivate_auc),
        "results_by_target_epsilon": [
            {"target_epsilon": te, "actual_epsilon": ae, "test_auc": float(a)}
            for te, ae, a in privacy_results
        ],
    },
    "byzantine_robustness": {
        "num_institutions": K,
        "malicious_scale": MALICIOUS_SCALE,
        "honest_test_auc": float(honest_auc),
        "attacked_naive_mean_test_auc": float(attacked_auc),
        "attacked_median_defended_test_auc": float(median_auc),
        "attacked_trimmed_mean_defended_test_auc": float(trimmed_auc),
    },
    "fairness_under_federation": {
        "centralized": {"test_auc": float(central_auc2), "demographic_parity_difference": float(central_dpd), "equalized_odds_difference": float(central_eod)},
        "federated_iid": {"test_auc": float(fed_auc2), "demographic_parity_difference": float(fed_dpd), "equalized_odds_difference": float(fed_eod)},
    },
    "random_seed": SEED,
    "known_limitations": [
        "5 simulated institutions on an 800-row dataset -- small-scale, illustrative federation, not a production-scale deployment.",
        "DP-SGD results vary run-to-run due to injected noise; a single seed per epsilon here, not averaged across multiple seeds.",
        "Byzantine-robust aggregation (median, trimmed mean) assumes malicious clients are a MINORITY -- neither defends against a colluding majority of institutions.",
        "The fairness-under-federation check (Lesson 8) is a single comparison, not a systematic study across many federated configurations.",
    ],
}

metrics_path = Path("artifacts/module7_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("Saved:", metrics_path.resolve())
print(json.dumps(metrics_summary, indent=2))